# NefroQuest — Revisão de Questões (gpt-5.4-mini)

**3 passos:**
1. Cole sua OpenAI API key e rode a Célula 1
2. Faça upload dos 3 arquivos e rode a Célula 2
3. Rode a Célula 3 e aguarde — ao final o `index.html` revisado será baixado automaticamente

In [ ]:
# ══════════════════════════════════════════════════════════
# PASSO 1 — Cole sua API key aqui e rode esta célula
# ══════════════════════════════════════════════════════════
OPENAI_API_KEY = "sk-..."  # ← substitua pela sua chave

# Não mexa no resto
import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
!pip install -q openai
print("✓ Pronto. Rode o Passo 2.")

In [ ]:
# ══════════════════════════════════════════════════════════
# PASSO 2 — Faça upload dos 3 arquivos e rode esta célula
# Arquivos necessários:
#   • index.html
#   • review_questions.py
#   • apply_reviews.py
# ══════════════════════════════════════════════════════════
from google.colab import files

print("Selecione os 3 arquivos para upload (index.html, review_questions.py, apply_reviews.py)")
uploaded = files.upload()

# Verifica se os 3 arquivos foram carregados
needed = {"index.html", "review_questions.py", "apply_reviews.py"}
missing = needed - set(uploaded.keys())
if missing:
    print(f"\n⚠ Arquivo(s) faltando: {missing}")
    print("Rode esta célula novamente e selecione todos os 3 arquivos.")
else:
    import re
    with open("index.html", encoding="utf-8") as f:
        html = f.read()
    bloco = html[html.find("const topics = ["):html.find("\n];", html.find("const topics = ["))]
    n = len(re.findall(r'"t":', bloco))
    print(f"\n✓ Uploads OK. Questões encontradas: {n}")
    print("Rode o Passo 3.")

In [ ]:
# ══════════════════════════════════════════════════════════
# PASSO 3 — Roda todos os lotes e baixa o index.html final
# Modelo: gpt-5.4-mini (escalonamento automático para gpt-5.4)
# Tempo estimado: ~60-90 min para 1034 questões
# Se o Colab desconectar, rode esta célula novamente —
# os lotes já concluídos serão pulados automaticamente.
# ══════════════════════════════════════════════════════════
import subprocess, json, os, shutil
from google.colab import files

TOTAL    = 1034
BATCH_SZ = 100

batches = []
s = 0
while s < TOTAL:
    batches.append((s, min(s + BATCH_SZ, TOTAL)))
    s += BATCH_SZ

total_ok  = 0
total_err = 0

for i, (s, e) in enumerate(batches):
    out = f"reviewed_{s}_{e}.json"

    # ── Checkpoint: pula lote já feito ──────────────────
    if os.path.exists(out):
        with open(out) as f_:
            data = json.load(f_)
        ok = sum(1 for r in data if not r.get("_error"))
        print(f"[Lote {i+1:02d}/{len(batches)}] {s}–{e}: JÁ FEITO ({ok}/{e-s} OK) — pulando")
        total_ok  += ok
        total_err += (e - s) - ok
        continue

    # ── Revisão via gpt-5.4-mini ─────────────────────────
    print(f"\n[Lote {i+1:02d}/{len(batches)}] Revisando questões {s}–{e-1}...")
    r = subprocess.run(
        ["python", "review_questions.py",
         "--start", str(s), "--end", str(e), "--out", out]
    )

    if r.returncode != 0 or not os.path.exists(out):
        print(f"  ERRO no lote {i+1}. Verifique a API key e rode novamente.")
        break

    with open(out) as f_:
        data = json.load(f_)
    ok  = sum(1 for r_ in data if not r_.get("_error"))
    err = len(data) - ok
    total_ok  += ok
    total_err += err

    # ── Aplica ao index.html ─────────────────────────────
    subprocess.run(["python", "apply_reviews.py", out])

    # Backup de segurança
    shutil.copy("index.html", f"index_backup_lote{i+1:02d}.html")

    print(f"  → {ok} revisadas OK | {err} com erro")

# ── Relatório ────────────────────────────────────────────
print(f"""
════════════════════════════════
CONCLUÍDO
  Revisadas com sucesso : {total_ok}
  Com erro de API       : {total_err}
  index.html final      : {os.path.getsize('index.html'):,} bytes
════════════════════════════════
Baixando index.html...
""")
files.download("index.html")

---
## Após o download

Substitua o `index.html` no seu repositório local pelo arquivo baixado e faça o commit:

```bash
git add index.html
git commit -m "feat: revisar 1034 questões via gpt-5.4-mini"
git push
```

## Se houver erros de API em algumas questões

Rode esta célula extra para retentar apenas as que falharam:

```python
import json, os, time, importlib.util, subprocess
from openai import OpenAI

spec = importlib.util.spec_from_file_location('rq', 'review_questions.py')
rq   = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rq)
client  = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
all_q   = rq.extract_questions('index.html')
TOTAL, BATCH_SZ = 1034, 100
batches = [(s, min(s+BATCH_SZ, TOTAL)) for s in range(0, TOTAL, BATCH_SZ)]

for s, e in batches:
    fname = f'reviewed_{s}_{e}.json'
    if not os.path.exists(fname): continue
    with open(fname) as f: data = json.load(f)
    errs = [r for r in data if r.get('_error')]
    if not errs: continue
    print(f'Lote {s}–{e}: retentando {len(errs)} questões...')
    updated = {r['_idx']: r for r in data}
    for r in errs:
        idx = r['_idx']
        print(f'  idx={idx}...', end=' ')
        try:
            rev = rq.review_question(client, all_q[idx], idx, rq.DEFAULT_MODEL, rq.DEFAULT_EFFORT)
            rev['_t']    = all_q[idx].get('t', '')
            rev['_idx']  = idx
            rev['_diff'] = all_q[idx].get('diff', 'medium')
            rev['_cat']  = all_q[idx].get('cat', '')
            rev['_refs'] = all_q[idx].get('refs', [])
            rev['_model'] = rq.DEFAULT_MODEL
            updated[idx] = rev
            print('OK')
        except Exception as ex:
            print(f'ERRO: {ex}')
        time.sleep(2)
    new_data = sorted(updated.values(), key=lambda x: x['_idx'])
    with open(fname, 'w', encoding='utf-8') as f: json.dump(new_data, f, ensure_ascii=False, indent=2)
    subprocess.run(['python', 'apply_reviews.py', fname])
print('Retentativa concluída.')
```